<div style="background: linear-gradient(135deg, #1B3A5C 0%, #1F7A8C 100%);
                   padding: 40px 30px; border-radius: 12px; margin-bottom: 10px;">
  <h1 style="color: white; font-size: 2.2em; margin: 0 0 10px 0;">
    🛒 Store Analytics: Performance Assessment Metrics
  </h1>
  <h3 style="color: #a8d8e8; margin: 0 0 16px 0; font-weight: normal;">
    A Hands-On Python Notebook · RetailCo Churn Prediction
  </h3>
  <hr style="border-color: #E07B39; border-width: 2px; margin: 16px 0;">
  <p style="color: #ddeeff; font-size: 1.05em; margin: 0;">
    <strong>Topics:</strong> Holdout Validation &nbsp;·&nbsp;
    K-Fold Cross-Validation &nbsp;·&nbsp;
    ROC Curve &nbsp;·&nbsp;
    AUC &nbsp;·&nbsp;
    Bias–Variance Tradeoff &nbsp;·&nbsp;
    🎯 Student Challenge
  </p>
</div>


## 📚 Learning Objectives

By completing this notebook you will be able to:

1. **Generate** a realistic retail churn dataset and understand its structure
2. **Apply** Holdout and K-Fold Cross-Validation to estimate model performance honestly
3. **Plot and interpret** ROC curves for multiple classifiers
4. **Compute and compare** AUC scores and explain what they mean in business terms
5. **Diagnose** Bias and Variance problems using learning curves
6. **Solve** an independent challenge applying all five concepts to a new scenario

> **Scenario:** You are a Data Scientist at *RetailCo*, a national grocery chain.  
> Your task: build and evaluate a model that predicts **customer churn** (will a customer stop shopping in the next 90 days?).  
> A churn rate of **~15%** makes this a classic *imbalanced binary classification* problem.


## ⚙️ Section 0 — Environment Setup

Run this cell first. It imports all required libraries and sets a consistent visual style.

In [ ]:
# ── Core libraries ──────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ── Scikit-learn ─────────────────────────────────────────────────
from sklearn.datasets import make_classification
from sklearn.model_selection import (
    train_test_split, KFold, StratifiedKFold,
    cross_val_score, learning_curve
)
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    roc_curve, auc, roc_auc_score,
    confusion_matrix, classification_report,
    accuracy_score, ConfusionMatrixDisplay
)
from sklearn.pipeline import Pipeline

# ── Visual style ─────────────────────────────────────────────────
NAVY   = '#1B3A5C'
TEAL   = '#1F7A8C'
ORANGE = '#E07B39'
GREEN  = '#27AE60'
RED    = '#E74C3C'
LGRAY  = '#F4F6F7'

plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor':   '#F8FAFB',
    'axes.spines.top':  False,
    'axes.spines.right': False,
    'axes.grid':        True,
    'grid.alpha':       0.4,
    'font.size':        11,
    'axes.titlesize':   13,
    'axes.labelsize':   11,
})

np.random.seed(42)
print("✅ All libraries loaded successfully!")
print(f"   NumPy {np.__version__} | Pandas {pd.__version__} | Scikit-learn: ready")


---
## 🗂️ Section 1 — The RetailCo Dataset

We simulate a realistic **customer churn dataset** with features a grocery chain would actually have.  
The target variable `churned` = 1 if the customer stopped shopping within 90 days.


In [ ]:
# ── Synthetic RetailCo customer dataset ──────────────────────────
def generate_retailco_data(n=5000, churn_rate=0.15, random_state=42):
    """
    Generates a realistic retail customer churn dataset.
    
    Features:
        avg_weekly_spend   : average £ spent per week (last 6 months)
        store_visits_month : average visits per month
        basket_diversity   : # unique product categories per basket
        distance_km        : distance from nearest store (km)
        days_since_purchase: recency — days since last visit
        loyalty_years      : years as a loyalty-card member
        promo_response_rate: fraction of promotions the customer responded to
        age_group          : encoded age bracket (0=18-25, 1=26-40, 2=41-60, 3=60+)
    
    Target:
        churned: 1 = churned within 90 days, 0 = still active
    """
    rng = np.random.RandomState(random_state)
    n_churn = int(n * churn_rate)
    n_active = n - n_churn

    def make_group(size, is_churn):
        return {
            'avg_weekly_spend':    rng.normal(25 if is_churn else 65, 15, size).clip(5, 200),
            'store_visits_month':  rng.normal(1.5 if is_churn else 4.5, 1.2, size).clip(0, 12),
            'basket_diversity':    rng.normal(3 if is_churn else 7, 1.5, size).clip(1, 15),
            'distance_km':         rng.exponential(8 if is_churn else 3, size).clip(0.5, 30),
            'days_since_purchase': rng.normal(55 if is_churn else 12, 18, size).clip(1, 120),
            'loyalty_years':       rng.exponential(1.2 if is_churn else 3.5, size).clip(0, 15),
            'promo_response_rate': rng.beta(1, 5 if is_churn else 2, size),
            'age_group':           rng.randint(0, 4, size),
            'churned':             np.ones(size, int) if is_churn else np.zeros(size, int),
        }

    df = pd.concat([
        pd.DataFrame(make_group(n_churn,  is_churn=True)),
        pd.DataFrame(make_group(n_active, is_churn=False)),
    ]).sample(frac=1, random_state=random_state).reset_index(drop=True)

    return df

df = generate_retailco_data(n=5000)

print("=" * 55)
print("  RetailCo Customer Dataset — Summary")
print("=" * 55)
print(f"  Total customers   : {len(df):,}")
print(f"  Churned (1)       : {df['churned'].sum():,}  ({df['churned'].mean()*100:.1f}%)")
print(f"  Active  (0)       : {(df['churned']==0).sum():,}  ({(df['churned']==0).mean()*100:.1f}%)")
print(f"  Features          : {df.shape[1]-1}")
print("=" * 55)
df.head(8)


In [ ]:
# ── Exploratory visualisation ────────────────────────────────────
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
fig.suptitle('RetailCo Dataset — Feature Distributions by Churn Status', 
             fontsize=14, fontweight='bold', color=NAVY, y=1.01)

features = ['avg_weekly_spend', 'store_visits_month', 'basket_diversity',
            'distance_km', 'days_since_purchase', 'loyalty_years',
            'promo_response_rate', 'age_group']

labels = ['Avg Weekly Spend (£)', 'Store Visits/Month', 'Basket Diversity',
          'Distance (km)', 'Days Since Purchase', 'Loyalty Years',
          'Promo Response Rate', 'Age Group']

for ax, feat, label in zip(axes.flat, features, labels):
    for val, color, name in [(0, TEAL, 'Active'), (1, ORANGE, 'Churned')]:
        subset = df[df['churned'] == val][feat]
        ax.hist(subset, bins=25, alpha=0.6, color=color, label=name, density=True)
    ax.set_title(label, fontweight='bold', color=NAVY)
    ax.set_xlabel('')

axes[0, 0].legend(fontsize=9)
plt.tight_layout()
plt.show()
print("📊 Observation: Churned customers spend less, visit less, live farther away,")
print("   and have higher 'days since purchase'. These are strong predictive signals.")


---
## 🔀 Section 2 — Holdout Validation

### Concept
Split the dataset **once** into Training and Test sets. Train on training data, evaluate on test data.  
The test set simulates "future customers" the model has never seen.

**Key rule:** Never use the test set during model development — only for the *final* evaluation.

### Formula
```
Total Data  →  Training Set (80%)  +  Test Set (20%)
```


In [ ]:
# ── Prepare features and target ──────────────────────────────────
FEATURES = ['avg_weekly_spend', 'store_visits_month', 'basket_diversity',
            'distance_km', 'days_since_purchase', 'loyalty_years',
            'promo_response_rate', 'age_group']
TARGET = 'churned'

X = df[FEATURES].values
y = df[TARGET].values

# ── Holdout Split ────────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,          # 20% held out
    stratify=y,              # preserve churn ratio in both splits
    random_state=42
)

print("=" * 50)
print("  Holdout Split Results")
print("=" * 50)
print(f"  Training samples : {len(X_train):,}  ({len(X_train)/len(X)*100:.0f}%)")
print(f"  Test samples     : {len(X_test):,}   ({len(X_test)/len(X)*100:.0f}%)")
print(f"  Train churn rate : {y_train.mean()*100:.1f}%")
print(f"  Test churn rate  : {y_test.mean()*100:.1f}%  ← stratify keeps it balanced")
print("=" * 50)


In [ ]:
# ── Visualise the holdout split ───────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle('Holdout Validation — RetailCo (5,000 customers)', 
             fontweight='bold', color=NAVY, fontsize=13)

# Left: stacked bar showing split
ax = axes[0]
splits = ['Training Set\n(80% = 4,000)', 'Test Set\n(20% = 1,000)']
active = [int((y_train==0).sum()), int((y_test==0).sum())]
churn  = [int((y_train==1).sum()), int((y_test==1).sum())]
x_pos = [0, 1]
bars_a = ax.bar(x_pos, active, color=TEAL,   label='Active (0)',  width=0.5)
bars_c = ax.bar(x_pos, churn,  color=ORANGE, label='Churned (1)', width=0.5, bottom=active)
ax.set_xticks(x_pos); ax.set_xticklabels(splits)
ax.set_ylabel('Number of Customers')
ax.set_title('Dataset Split', fontweight='bold', color=NAVY)
ax.legend()
for bar, n in zip(bars_a, active):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()/2,
            f'{n:,}', ha='center', va='center', color='white', fontweight='bold', fontsize=10)
for bar, a, c in zip(bars_c, active, churn):
    ax.text(bar.get_x()+bar.get_width()/2, a + c/2,
            f'{c:,}', ha='center', va='center', color='white', fontweight='bold', fontsize=10)

# Right: timeline metaphor
ax2 = axes[1]
ax2.set_xlim(0, 10); ax2.set_ylim(0, 3); ax2.axis('off')
ax2.add_patch(plt.Rectangle((0.2, 1.2), 7.7, 0.9, color=TEAL,   alpha=0.85, transform=ax2.transData))
ax2.add_patch(plt.Rectangle((7.9, 1.2), 1.9, 0.9, color=ORANGE, alpha=0.85, transform=ax2.transData))
ax2.text(4.05, 1.65, 'TRAINING SET  →  Model Learns Churn Patterns',
         ha='center', va='center', color='white', fontweight='bold', fontsize=10.5)
ax2.text(8.85, 1.65, 'TEST
SET', ha='center', va='center', color='white', fontweight='bold', fontsize=9)
ax2.text(5, 0.7, 'Rule: Test set is sealed — only opened for final evaluation!',
         ha='center', color=RED, fontsize=10, fontstyle='italic', fontweight='bold')
ax2.set_title('The "Sealed Envelope" Principle', fontweight='bold', color=NAVY, fontsize=12)

plt.tight_layout()
plt.show()


In [ ]:
# ── Train + evaluate on holdout ──────────────────────────────────
# We use a pipeline: StandardScaler → Logistic Regression
lr_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(max_iter=1000, random_state=42))
])

lr_pipeline.fit(X_train, y_train)

# Predictions
y_pred       = lr_pipeline.predict(X_test)
y_prob       = lr_pipeline.predict_proba(X_test)[:, 1]
holdout_acc  = accuracy_score(y_test, y_pred)
holdout_auc  = roc_auc_score(y_test, y_prob)

print("=" * 50)
print("  Holdout Evaluation — Logistic Regression")
print("=" * 50)
print(f"  Accuracy : {holdout_acc*100:.2f}%")
print(f"  AUC      : {holdout_auc:.4f}")
print()
print("  Classification Report:")
print(classification_report(y_test, y_pred, target_names=['Active', 'Churned']))


In [ ]:
# ── Confusion matrix ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
fig.suptitle('Holdout Results — Logistic Regression on RetailCo Test Set',
             fontweight='bold', color=NAVY, fontsize=13)

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=['Active', 'Churned'])
disp.plot(ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title('Confusion Matrix', fontweight='bold', color=NAVY)

# Business interpretation
ax2 = axes[1]; ax2.axis('off')
tn, fp, fn, tp = cm.ravel()
total = tn+fp+fn+tp
rows = [
    ('True Positives (TP)',  tp, 'Churners correctly flagged → targeted with voucher', GREEN),
    ('False Positives (FP)', fp, 'Loyal customers wrongly flagged → wasted vouchers', ORANGE),
    ('False Negatives (FN)', fn, 'Churners we MISSED → lost customers', RED),
    ('True Negatives (TN)',  tn, 'Correctly identified loyal customers', TEAL),
]
y_pos = 0.85
for label, count, meaning, color in rows:
    ax2.text(0.0, y_pos, f'● {label}: {count}', fontsize=10.5,
             fontweight='bold', color=color, transform=ax2.transAxes)
    ax2.text(0.05, y_pos-0.065, meaning, fontsize=9.5,
             color='#444', transform=ax2.transAxes, fontstyle='italic')
    y_pos -= 0.19

ax2.text(0.0, 0.07,
         f'💡 Accuracy = {holdout_acc*100:.1f}% but churn recall = {tp/(tp+fn)*100:.1f}% — '
         f'we catch only {tp/(tp+fn)*100:.0f}% of churners!',
         fontsize=9.5, color=NAVY, transform=ax2.transAxes, fontweight='bold')
ax2.set_title('Business Interpretation', fontweight='bold', color=NAVY)

plt.tight_layout()
plt.show()

print(f"\n⚠️  Holdout warning: This is ONE split. A different random_state could give")
print(f"   different results. That's why we use K-Fold next!")


---
## 🔁 Section 3 — K-Fold Cross-Validation

### Concept
Instead of one split, we make **K splits** and average the results.  
Every sample is used for validation exactly **once** — far more reliable than holdout.

### Why this matters for RetailCo
Churn patterns vary by season and store region. K-Fold ensures no single evaluation  
is accidentally lucky or unlucky based on which customers ended up in the test set.

```
Round 1: [Fold 1 = VAL] [Fold 2] [Fold 3] [Fold 4] [Fold 5]
Round 2: [Fold 1] [Fold 2 = VAL] [Fold 3] [Fold 4] [Fold 5]
...
Final Score = mean(AUC_round1, AUC_round2, ..., AUC_round5)
```


In [ ]:
# ── Stratified 5-Fold Cross-Validation ──────────────────────────
# StratifiedKFold ensures each fold has the same churn ratio (~15%)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

lr_pipeline_cv = Pipeline([
    ('scaler', StandardScaler()),
    ('clf',    LogisticRegression(max_iter=1000, random_state=42))
])

# cross_val_score handles the split, train, evaluate loop for us
cv_auc_scores = cross_val_score(
    lr_pipeline_cv, X, y,
    cv=skf,
    scoring='roc_auc',
    n_jobs=-1
)

print("=" * 58)
print("  5-Fold Stratified Cross-Validation — Logistic Regression")
print("=" * 58)
for i, score in enumerate(cv_auc_scores, 1):
    bar = '█' * int(score * 40)
    print(f"  Fold {i}: AUC = {score:.4f}  {bar}")
print("  " + "─" * 54)
print(f"  Mean AUC : {cv_auc_scores.mean():.4f}")
print(f"  Std  AUC : {cv_auc_scores.std():.4f}  ← lower = more stable model")
print(f"  95% CI   : [{cv_auc_scores.mean()-2*cv_auc_scores.std():.4f}, "
      f"{cv_auc_scores.mean()+2*cv_auc_scores.std():.4f}]")
print("=" * 58)


In [ ]:
# ── Compare multiple models with K-Fold ──────────────────────────
models = {
    'Dummy (Baseline)':      DummyClassifier(strategy='most_frequent'),
    'Logistic Regression':   Pipeline([('sc', StandardScaler()),
                                       ('clf', LogisticRegression(max_iter=1000, random_state=42))]),
    'Decision Tree (deep)':  DecisionTreeClassifier(max_depth=20, random_state=42),
    'Decision Tree (pruned)':DecisionTreeClassifier(max_depth=5,  random_state=42),
    'Random Forest':         RandomForestClassifier(n_estimators=150, max_depth=8, random_state=42),
    'Gradient Boosting':     GradientBoostingClassifier(n_estimators=100, max_depth=4, random_state=42),
}

results = {}
print("Running 5-Fold CV on all models...")
for name, model in models.items():
    scores = cross_val_score(model, X, y, cv=skf, scoring='roc_auc', n_jobs=-1)
    results[name] = scores
    print(f"  ✓ {name:<28} AUC = {scores.mean():.4f} ± {scores.std():.4f}")

print("\n✅ Done!")


In [ ]:
# ── Visualise K-Fold comparison ───────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
fig.suptitle('K-Fold Cross-Validation — Model Comparison (RetailCo Churn)',
             fontweight='bold', color=NAVY, fontsize=13)

names   = list(results.keys())
means   = [results[n].mean() for n in names]
stds    = [results[n].std()  for n in names]
colors  = [RED, TEAL, ORANGE, '#F1C40F', GREEN, NAVY]

# Left: bar chart with error bars
ax = axes[0]
bars = ax.barh(names, means, xerr=stds, color=colors, alpha=0.85,
               error_kw=dict(elinewidth=2, capsize=5, ecolor='#555'))
ax.axvline(0.5, color=RED, lw=1.5, ls='--', label='Random baseline (AUC=0.5)')
ax.set_xlabel('Mean AUC (5-Fold)')
ax.set_title('Mean AUC ± Std Dev', fontweight='bold', color=NAVY)
ax.legend(fontsize=9)
for bar, mean in zip(bars, means):
    ax.text(mean + 0.005, bar.get_y() + bar.get_height()/2,
            f'{mean:.3f}', va='center', fontsize=9, fontweight='bold')

# Right: box plot of fold-by-fold scores
ax2 = axes[1]
data_for_box = [results[n] for n in names]
bp = ax2.boxplot(data_for_box, vert=False, patch_artist=True,
                 medianprops=dict(color='white', linewidth=2))
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color); patch.set_alpha(0.8)
ax2.set_yticks(range(1, len(names)+1))
ax2.set_yticklabels(names, fontsize=9.5)
ax2.set_xlabel('AUC Score per Fold')
ax2.set_title('Score Distribution Across Folds\n(spread = variance)', fontweight='bold', color=NAVY)
ax2.axvline(0.5, color=RED, lw=1.5, ls='--', alpha=0.7)

plt.tight_layout()
plt.show()

best_model = names[means.index(max(means))]
print(f"\n🏆 Best model by mean AUC: {best_model}")
print(f"   Note the deep Decision Tree has HIGH variance (wide box) — a sign of overfitting!")


---
## 📈 Section 4 — ROC Curve

### Concept
The **ROC curve** plots **True Positive Rate (Recall)** vs **False Positive Rate** at every possible classification threshold.

| Metric | Formula | RetailCo Meaning |
|--------|---------|-----------------|
| **TPR / Recall** | TP / (TP + FN) | % of churners we correctly catch |
| **FPR** | FP / (FP + TN) | % of loyal customers we wrongly flag |
| **Threshold** | Model's probability cutoff | e.g., flag as churner if P(churn) > 0.40 |

### Business Trade-off
- **Low threshold** → catch more churners (high TPR) but waste more vouchers (high FPR)
- **High threshold** → fewer false alarms but miss more churners
- The ROC curve makes this trade-off **visible for every possible threshold at once**


In [ ]:
# ── Train all models on train split and plot ROC curves ──────────
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

roc_models = {
    'Logistic Regression':    Pipeline([('sc', StandardScaler()),
                                        ('clf', LogisticRegression(max_iter=1000, random_state=42))]),
    'Decision Tree (pruned)': DecisionTreeClassifier(max_depth=5, random_state=42),
    'Random Forest':          RandomForestClassifier(n_estimators=150, max_depth=8, random_state=42),
    'Gradient Boosting':      GradientBoostingClassifier(n_estimators=100, max_depth=4, random_state=42),
    'Dummy Baseline':         DummyClassifier(strategy='most_frequent'),
}

roc_colors  = [TEAL, ORANGE, GREEN, NAVY, LGRAY]
roc_results = {}

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
fig.suptitle('ROC Curves — RetailCo Churn Models', fontweight='bold', color=NAVY, fontsize=14)

ax = axes[0]
for (name, model), color in zip(roc_models.items(), roc_colors):
    model.fit(X_tr, y_tr)
    probs = model.predict_proba(X_te)[:, 1]
    fpr, tpr, thresholds = roc_curve(y_te, probs)
    auc_score = auc(fpr, tpr)
    roc_results[name] = (fpr, tpr, thresholds, auc_score)
    ls = '--' if name == 'Dummy Baseline' else '-'
    lw = 1.5 if name == 'Dummy Baseline' else 2.3
    ax.plot(fpr, tpr, color=color, lw=lw, ls=ls,
            label=f'{name}  (AUC={auc_score:.3f})')

ax.plot([0,1],[0,1], 'k--', lw=1, alpha=0.4, label='Random (AUC=0.500)')
ax.fill_between(*roc_results['Gradient Boosting'][:2], alpha=0.07, color=NAVY)
ax.set_xlabel('False Positive Rate (FPR)
← Fewer false alarms          More false alarms →')
ax.set_ylabel('True Positive Rate (TPR)
← Fewer churners caught     More churners caught →')
ax.set_title('ROC Curves — All Models', fontweight='bold', color=NAVY)
ax.legend(fontsize=9, loc='lower right')
ax.set_xlim([-0.01, 1.01]); ax.set_ylim([-0.01, 1.05])

# ── Right: Threshold analysis for best model (Gradient Boosting) ─
ax2 = axes[1]
fpr_gb, tpr_gb, thresh_gb, _ = roc_results['Gradient Boosting']

# Remove inf threshold
mask = thresh_gb < np.inf
fpr_gb_m   = fpr_gb[:-1][mask[:-1]]   # roc_curve returns len+1 for thresholds
tpr_gb_m   = tpr_gb[:-1][mask[:-1]]
thresh_gb_m = thresh_gb[mask]

ax2.plot(thresh_gb_m, tpr_gb_m, color=GREEN,  lw=2.5, label='TPR (Recall) — catch churners')
ax2.plot(thresh_gb_m, fpr_gb_m, color=ORANGE, lw=2.5, label='FPR — false alarms on loyal customers')
ax2.axvline(0.40, color=NAVY, lw=1.8, ls=':', label='Threshold = 0.40 (chosen)')
ax2.axvline(0.50, color=RED,  lw=1.5, ls=':', alpha=0.5, label='Threshold = 0.50')
ax2.set_xlabel('Classification Threshold  (P(churn) >  threshold → flag as churner)')
ax2.set_ylabel('Rate')
ax2.set_title('Gradient Boosting: TPR & FPR vs Threshold\n(choose threshold for your budget!)',
              fontweight='bold', color=NAVY)
ax2.legend(fontsize=9)
ax2.set_xlim([0, 1])

plt.tight_layout()
plt.show()


In [ ]:
# ── Threshold decision analysis ──────────────────────────────────
print("=" * 65)
print("  Gradient Boosting — Business Impact at Different Thresholds")
print("=" * 65)
print(f"  {'Threshold':>10} | {'TPR':>6} | {'FPR':>6} | {'Churners Caught':>16} | {'Wasted Vouchers':>15}")
print("  " + "─" * 63)

gb_model = roc_models['Gradient Boosting']
y_prob_gb = gb_model.predict_proba(X_te)[:, 1]
n_churners = y_te.sum()
n_active   = (y_te == 0).sum()

for thresh in [0.20, 0.30, 0.40, 0.45, 0.50, 0.60, 0.70]:
    y_pred_t = (y_prob_gb >= thresh).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_te, y_pred_t).ravel()
    tpr_t = tp / (tp + fn)
    fpr_t = fp / (fp + tn)
    marker = ' ◀ recommended' if thresh == 0.40 else ''
    print(f"  {thresh:>10.2f} | {tpr_t:>6.3f} | {fpr_t:>6.3f} | "
          f"{tp:>8} / {n_churners:<6} | {fp:>8} / {n_active:<5}{marker}")

print("=" * 65)
print("\n💡 At threshold = 0.40:")
print("   → We catch ~80% of churners with only ~10% false alarms")
print("   → Balance: enough churners caught to justify voucher costs")


---
## 📊 Section 5 — AUC: Area Under the ROC Curve

### Concept
**AUC** = the area under the ROC curve, ranging from 0 to 1.

> 🎯 **Intuition:** If you randomly pick one churner and one loyal customer from your test set,  
> AUC = probability that the model gives the churner a **higher churn score** than the loyal customer.

### Why AUC over Accuracy?
- **Accuracy** is fooled by class imbalance (15% churn → 85% accuracy by always predicting "no churn")
- **AUC** is threshold-independent and handles imbalance naturally


In [ ]:
# ── AUC comparison dashboard ─────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5.5))
fig.suptitle('AUC Analysis — RetailCo Churn Prediction', fontweight='bold', color=NAVY, fontsize=14)

# ── Panel 1: AUC scores bar ──────────────────────────────────────
ax = axes[0]
model_names = [n for n in roc_results if n != 'Dummy Baseline']
auc_scores  = [roc_results[n][3] for n in model_names]
bar_colors  = [TEAL, ORANGE, GREEN, NAVY]

bars = ax.bar(range(len(model_names)), auc_scores, color=bar_colors, alpha=0.85, width=0.6)
ax.axhline(0.5, color=RED, lw=1.5, ls='--', label='Random baseline')
ax.axhline(roc_results['Dummy Baseline'][3], color='gray', lw=1, ls=':', alpha=0.6)
ax.set_xticks(range(len(model_names)))
ax.set_xticklabels([n.replace(' ', '\n') for n in model_names], fontsize=9)
ax.set_ylabel('AUC Score')
ax.set_title('AUC Comparison', fontweight='bold', color=NAVY)
ax.set_ylim([0.45, 1.02])
ax.legend(fontsize=9)
for bar, score in zip(bars, auc_scores):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
            f'{score:.3f}', ha='center', fontsize=10, fontweight='bold')

# ── Panel 2: AUC = probability demo ─────────────────────────────
ax2 = axes[1]
churners    = y_prob_gb[y_te == 1]
non_churners= y_prob_gb[y_te == 0]
ax2.hist(non_churners, bins=35, alpha=0.65, color=TEAL,   label='Active customers',  density=True)
ax2.hist(churners,     bins=35, alpha=0.65, color=ORANGE, label='Churned customers',  density=True)
ax2.axvline(0.40, color=NAVY, lw=2, ls='--', label='Threshold = 0.40')
ax2.set_xlabel('Predicted Churn Probability')
ax2.set_ylabel('Density')
ax2.set_title('Score Distributions\n(good model = well separated peaks)', fontweight='bold', color=NAVY)
ax2.legend(fontsize=9)
ax2.text(0.5, 0.92, f'AUC = {roc_results["Gradient Boosting"][3]:.3f}',
         transform=ax2.transAxes, fontsize=13, fontweight='bold', color=NAVY, ha='center')

# ── Panel 3: Accuracy vs AUC trap ───────────────────────────────
ax3 = axes[2]
model_names_all = list(roc_results.keys())
acc_scores, auc_scores_all = [], []
for name, (fpr_r, tpr_r, _, auc_r) in roc_results.items():
    model = roc_models[name]
    y_pred_tmp = model.predict(X_te)
    acc_scores.append(accuracy_score(y_te, y_pred_tmp))
    auc_scores_all.append(auc_r)

sc = ax3.scatter(acc_scores, auc_scores_all,
                 c=[TEAL, ORANGE, GREEN, NAVY, RED], s=160, zorder=5, edgecolors='white', lw=1.5)
for i, name in enumerate(model_names_all):
    ax3.annotate(name.replace(' ', '\n'), (acc_scores[i], auc_scores_all[i]),
                 textcoords='offset points', xytext=(6, 4), fontsize=7.5)
ax3.set_xlabel('Accuracy')
ax3.set_ylabel('AUC')
ax3.set_title('Accuracy vs AUC\n(Dummy has 85% accuracy but AUC≈0.5!)', fontweight='bold', color=NAVY)
ax3.axhline(0.5, color=RED, lw=1, ls='--', alpha=0.5)

plt.tight_layout()
plt.show()

print("\n⚠️  The Dummy classifier has 85% accuracy (always predicts 'no churn')!")
print("   But AUC = 0.50 — it's completely useless. Never rely on accuracy alone!")


---
## ⚖️ Section 6 — Bias–Variance Tradeoff

### The Fundamental Tension

| | **High Bias (Underfitting)** | **High Variance (Overfitting)** |
|---|---|---|
| **Cause** | Model too simple | Model too complex |
| **Train score** | Low | Very high |
| **Val score** | Low | Much lower than train |
| **Gap** | Small | Large |
| **Fix** | Add features, more complex model | Regularise, prune, ensemble, more data |

### Error Decomposition
```
Total Error = Bias² + Variance + Irreducible Noise
```

### Learning Curves
Plot training and validation score as **training set size grows**.  
The shape tells you whether you have a bias or variance problem.


In [ ]:
# ── Bias-Variance U-curve ────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Bias–Variance Tradeoff — Conceptual & Empirical',
             fontweight='bold', color=NAVY, fontsize=13)

# Left: theoretical U-curve
ax = axes[0]
complexity = np.linspace(0.5, 10, 200)
bias2 = 0.45 * np.exp(-0.42 * complexity)
var   = 0.018 * np.exp(0.43 * complexity) - 0.018
total = bias2 + var + 0.05

ax.plot(complexity, bias2, color=TEAL,   lw=2.5, label='Bias²')
ax.plot(complexity, var,   color=ORANGE, lw=2.5, label='Variance')
ax.plot(complexity, total, color=NAVY,   lw=3,   ls='--', label='Total Error')
ax.fill_between(complexity, total, 0, alpha=0.06, color=NAVY)

opt = np.argmin(total)
ax.axvline(complexity[opt], color=NAVY, lw=1.5, ls=':', alpha=0.7)
ax.scatter([complexity[opt]], [total[opt]], color=NAVY, zorder=6, s=120)
ax.annotate('  Sweet Spot', xy=(complexity[opt], total[opt]),
            xytext=(complexity[opt]+0.8, total[opt]+0.04),
            fontsize=10, color=NAVY, fontweight='bold',
            arrowprops=dict(arrowstyle='-|>', color=NAVY))
ax.text(1.5, 0.38, '← Underfitting', fontsize=9, color=TEAL, fontstyle='italic')
ax.text(7.5, 0.38, 'Overfitting →',  fontsize=9, color=ORANGE, fontstyle='italic')
ax.set_xlabel('Model Complexity (e.g., tree depth, n_features)')
ax.set_ylabel('Error')
ax.set_title('Bias–Variance Curve (Theoretical)', fontweight='bold', color=NAVY)
ax.legend(fontsize=10); ax.set_ylim(0, 0.55); ax.set_xlim(0.5, 10)

# Right: AUC vs tree depth (empirical on RetailCo)
ax2 = axes[1]
depths = list(range(1, 21))
train_aucs, val_aucs = [], []
for d in depths:
    dt = DecisionTreeClassifier(max_depth=d, random_state=42)
    dt.fit(X_tr, y_tr)
    train_aucs.append(roc_auc_score(y_tr, dt.predict_proba(X_tr)[:,1]))
    cv_scores = cross_val_score(dt, X_tr, y_tr, cv=StratifiedKFold(5), scoring='roc_auc')
    val_aucs.append(cv_scores.mean())

ax2.plot(depths, train_aucs, color=TEAL,   lw=2.5, marker='o', ms=5, label='Training AUC')
ax2.plot(depths, val_aucs,   color=ORANGE, lw=2.5, marker='s', ms=5, label='CV Validation AUC')
ax2.fill_between(depths, train_aucs, val_aucs, alpha=0.12, color='gray', label='Bias–Variance gap')

best_depth = depths[np.argmax(val_aucs)]
ax2.axvline(best_depth, color=NAVY, lw=1.8, ls=':', label=f'Best depth = {best_depth}')
ax2.set_xlabel('Decision Tree Max Depth  (proxy for complexity)')
ax2.set_ylabel('AUC Score')
ax2.set_title(f'Empirical Bias–Variance on RetailCo\n(Decision Tree depth vs AUC)',
              fontweight='bold', color=NAVY)
ax2.legend(fontsize=9)

plt.tight_layout()
plt.show()
print(f"\n🌲 Best tree depth by CV-AUC: {best_depth}")
print(f"   At depth={best_depth}: train AUC = {train_aucs[best_depth-1]:.3f}, val AUC = {val_aucs[best_depth-1]:.3f}")
print(f"   At depth=20: train AUC = {train_aucs[-1]:.3f}, val AUC = {val_aucs[-1]:.3f} ← severe overfitting!")


In [ ]:
# ── Learning curves for 3 models ─────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Learning Curves — Diagnosing Bias & Variance on RetailCo Data',
             fontweight='bold', color=NAVY, fontsize=13)

lc_models = {
    'High Bias\n(Logistic Reg., 1 feature)': Pipeline([
        ('sc', StandardScaler()),
        ('clf', LogisticRegression(max_iter=500))
    ]),
    'High Variance\n(Decision Tree depth=20)': DecisionTreeClassifier(max_depth=20, random_state=42),
    'Well Calibrated\n(Random Forest depth=8)': RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42),
}

# For the "1 feature" logistic regression, select only recency
X_1feat = df[['days_since_purchase']].values
datasets = [X_1feat, X, X]  # different X per model

for ax, (name, model), X_lc in zip(axes, lc_models.items(), datasets):
    train_sizes, train_scores, val_scores = learning_curve(
        model, X_lc, y,
        train_sizes=np.linspace(0.1, 1.0, 12),
        cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
        scoring='roc_auc', n_jobs=-1
    )
    train_mean = train_scores.mean(axis=1)
    train_std  = train_scores.std(axis=1)
    val_mean   = val_scores.mean(axis=1)
    val_std    = val_scores.std(axis=1)

    ax.plot(train_sizes, train_mean, color=NAVY,   lw=2.5, label='Training AUC',   marker='o', ms=5)
    ax.plot(train_sizes, val_mean,   color=ORANGE, lw=2.5, label='Validation AUC', marker='s', ms=5)
    ax.fill_between(train_sizes, train_mean-train_std, train_mean+train_std, alpha=0.15, color=NAVY)
    ax.fill_between(train_sizes, val_mean-val_std,     val_mean+val_std,     alpha=0.15, color=ORANGE)
    ax.fill_between(train_sizes, train_mean, val_mean, alpha=0.08, color='gray', label='Gap')

    gap = (train_mean - val_mean)[-1]
    ax.set_title(f'{name}\nFinal gap = {gap:.3f}', fontweight='bold', color=NAVY, fontsize=10)
    ax.set_xlabel('Training Set Size')
    ax.set_ylabel('AUC Score')
    ax.set_ylim([0.45, 1.05])
    if ax == axes[0]:
        ax.legend(fontsize=8.5)

plt.tight_layout()
plt.show()

print("\n📊 Reading the learning curves:")
print("  High Bias:     Both curves are flat and LOW  → model can't capture complexity")
print("  High Variance: Large gap between train & val  → model memorises training data")
print("  Well Calibrated: Curves converge HIGH         → the sweet spot!")


In [ ]:
# ── Regularisation demo: fixing high variance ────────────────────
print("Fitting models with different regularisation strengths...")

C_values = [0.001, 0.01, 0.1, 1, 10, 100, 1000]
train_aucs_reg, val_aucs_reg = [], []

for C_val in C_values:
    lr = Pipeline([('sc', StandardScaler()),
                   ('clf', LogisticRegression(C=C_val, max_iter=1000, random_state=42))])
    lr.fit(X_tr, y_tr)
    train_aucs_reg.append(roc_auc_score(y_tr, lr.predict_proba(X_tr)[:,1]))
    scores = cross_val_score(lr, X_tr, y_tr,
                             cv=StratifiedKFold(5, shuffle=True, random_state=42),
                             scoring='roc_auc')
    val_aucs_reg.append(scores.mean())

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.semilogx(C_values, train_aucs_reg, color=NAVY,   lw=2.5, marker='o', label='Training AUC')
ax.semilogx(C_values, val_aucs_reg,   color=ORANGE, lw=2.5, marker='s', label='CV Validation AUC')
ax.fill_between(C_values, train_aucs_reg, val_aucs_reg, alpha=0.1, color='gray')

best_C = C_values[np.argmax(val_aucs_reg)]
ax.axvline(best_C, color=GREEN, lw=2, ls='--', label=f'Best C = {best_C}')
ax.set_xlabel('Regularisation Parameter C  (higher C = less regularisation = more complex)')
ax.set_ylabel('AUC')
ax.set_title('Logistic Regression — Regularisation vs Bias–Variance\n'
             'Low C = High Bias (underfitting)  |  High C = High Variance (overfitting)',
             fontweight='bold', color=NAVY)
ax.legend(fontsize=10)

plt.tight_layout()
plt.show()
print(f"\n🎯 Optimal regularisation C = {best_C}")
print(f"   Val AUC = {max(val_aucs_reg):.4f} at this setting")


---
## 🏁 Section 7 — Full Evaluation Pipeline Summary

Let's put everything together in one clean pipeline — the way you'd do it in a real RetailCo project.


In [ ]:
# ── Complete evaluation pipeline ─────────────────────────────────
print("=" * 65)
print("  RetailCo Churn Model — Complete Evaluation Pipeline")
print("=" * 65)

final_models = {
    'Logistic Regression': Pipeline([('sc', StandardScaler()),
                                     ('clf', LogisticRegression(C=1.0, max_iter=1000, random_state=42))]),
    'Random Forest':       RandomForestClassifier(n_estimators=150, max_depth=8, random_state=42),
    'Gradient Boosting':   GradientBoostingClassifier(n_estimators=100, max_depth=4, random_state=42),
}

summary = []
for name, model in final_models.items():
    # Step 1: K-Fold CV for model selection
    cv_scores = cross_val_score(model, X_train, y_train,
                                cv=StratifiedKFold(5, shuffle=True, random_state=42),
                                scoring='roc_auc', n_jobs=-1)
    # Step 2: Final evaluation on held-out test set
    model.fit(X_train, y_train)
    y_prob_f = model.predict_proba(X_test)[:,1]
    test_auc = roc_auc_score(y_test, y_prob_f)
    test_acc = accuracy_score(y_test, model.predict(X_test))

    summary.append({
        'Model':      name,
        'CV AUC (mean)': f"{cv_scores.mean():.4f}",
        'CV AUC (std)':  f"±{cv_scores.std():.4f}",
        'Test AUC':      f"{test_auc:.4f}",
        'Test Accuracy': f"{test_acc*100:.1f}%",
        'Verdict':       '✅ Good' if cv_scores.mean() > 0.80 else '⚠️ Marginal',
    })
    print(f"\n  {name}")
    print(f"    CV AUC:   {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")
    print(f"    Test AUC: {test_auc:.4f}")
    print(f"    Test Acc: {test_acc*100:.1f}%")

summary_df = pd.DataFrame(summary)
print("\n")
print(summary_df.to_string(index=False))
print("\n" + "=" * 65)
print("  RECOMMENDATION: Deploy Gradient Boosting (highest CV + Test AUC)")
print("  Use threshold = 0.40 for marketing campaign targeting")
print("=" * 65)


---
<div style="background: linear-gradient(135deg, #E07B39 0%, #1B3A5C 100%);
            padding: 30px; border-radius: 10px; margin: 20px 0;">
  <h1 style="color: white; margin: 0 0 8px 0;">🎯 Student Challenge</h1>
  <h3 style="color: #ffe0c0; font-weight: normal; margin: 0;">
    RetailCo Expansion — Online Store Churn Prediction
  </h3>
</div>

## Scenario

RetailCo has launched an **online grocery store**. Online shoppers behave differently from in-store customers:

- Higher churn rate: **~25%** (vs 15% in-store)  
- Different features matter: `app_sessions`, `delivery_rating`, `subscription_tier`  
- Smaller dataset: **2,000 customers** (vs 5,000)

Your manager says: *"Just use the in-store model — it's already built."*  
**Your job:** show why that's wrong, and build the correct evaluation pipeline.

---

## Your Tasks

Work through the cells below. Each has a `# TODO` comment where you write your code.  
Hints are provided — try without them first!

### Task 1 — Generate the Online Store Dataset
### Task 2 — Holdout Split with correct stratification  
### Task 3 — K-Fold CV with 5 folds, compare 3 models  
### Task 4 — Plot ROC curves and identify the best model  
### Task 5 — Find the optimal threshold for a budget of 300 vouchers  
### Task 6 — Plot learning curves and diagnose bias/variance  
### Task 7 — Written reflection questions  


### 📝 Task 1 — Generate the Online Store Dataset

Generate a dataset with **2,000 customers**, **~25% churn rate**, and the features listed below.

In [ ]:
# ─────────────────────────────────────────────────────
# TASK 1: Generate the online store dataset
# ─────────────────────────────────────────────────────
#
# Features to include:
#   app_sessions_month  : avg monthly app sessions  (churners: ~3, actives: ~12)
#   delivery_rating     : avg delivery rating 1-5   (churners: ~2.5, actives: ~4.2)
#   items_per_order     : avg items per order        (churners: ~8, actives: ~18)
#   subscription_tier   : 0=free, 1=basic, 2=premium (churners mostly tier 0)
#   days_since_last_order: recency                   (churners: ~40, actives: ~10)
#   cart_abandonment_rate: fraction of carts abandoned (churners: ~0.6, actives: ~0.2)
#   support_tickets_month: # customer support tickets (churners: ~2, actives: ~0.5)
#
# Target: churned (1 = churned, 0 = active)
# Use churn_rate = 0.25 and n = 2000
#
# HINT: Model your function on generate_retailco_data() above.
#       Use np.random.normal() and np.random.RandomState(seed)

def generate_online_data(n=2000, churn_rate=0.25, random_state=99):
    # TODO: implement this function
    pass

# Test your function:
df_online = generate_online_data()

# TODO: print the shape, churn rate, and first 5 rows
# Expected output:
#   Shape: (2000, 8)
#   Churn rate: ~25%


### 📝 Task 2 — Holdout Split

Split the online dataset into training (80%) and test (20%) sets. Make sure to **stratify** on the target.

In [ ]:
# ─────────────────────────────────────────────────────
# TASK 2: Holdout split
# ─────────────────────────────────────────────────────
ONLINE_FEATURES = ['app_sessions_month', 'delivery_rating', 'items_per_order',
                   'subscription_tier', 'days_since_last_order',
                   'cart_abandonment_rate', 'support_tickets_month']

# TODO: create X_online and y_online from df_online
X_online = None  # replace with correct code
y_online = None  # replace with correct code

# TODO: split into X_tr_o, X_te_o, y_tr_o, y_te_o
# Use test_size=0.20, stratify=y_online, random_state=42
X_tr_o, X_te_o, y_tr_o, y_te_o = None, None, None, None

# TODO: print the split sizes and churn rates in each split
# HINT: len(X_tr_o) should be 1600, len(X_te_o) should be 400
# HINT: y_tr_o.mean() should be close to 0.25


### 📝 Task 3 — K-Fold Cross-Validation

Run **5-fold stratified CV** on the online dataset. Test at least 3 models.

In [ ]:
# ─────────────────────────────────────────────────────
# TASK 3: K-Fold CV — compare 3+ models
# ─────────────────────────────────────────────────────
#
# Models to test (feel free to add more!):
#   1. Logistic Regression (with StandardScaler)
#   2. Random Forest
#   3. At least ONE more of your choice
#
# Use StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
# Use scoring='roc_auc'
#
# HINT: Loop over your models dict and use cross_val_score()

online_models = {
    # TODO: define your models here
}

online_cv_results = {}

# TODO: run CV for each model and store results
# Print mean AUC and std for each

# TODO: create a bar chart comparing the models
# BONUS: add error bars showing the std across folds


### 📝 Task 4 — ROC Curves

Plot ROC curves for all your models on the online test set. Identify the best model.

In [ ]:
# ─────────────────────────────────────────────────────
# TASK 4: ROC Curves
# ─────────────────────────────────────────────────────
#
# 1. Train each model on X_tr_o, y_tr_o
# 2. Get predicted probabilities on X_te_o
# 3. Compute fpr, tpr, auc using roc_curve() and auc()
# 4. Plot all curves on one figure
#
# HINT: roc_curve(y_te_o, probs)  → fpr, tpr, thresholds
# HINT: auc(fpr, tpr)              → scalar AUC score

fig, ax = plt.subplots(figsize=(8, 6))

# TODO: train each model, compute ROC, plot the curve

ax.plot([0,1],[0,1],'k--', alpha=0.4, label='Random baseline')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('Task 4 — ROC Curves: Online Store Churn Models', fontweight='bold', color=NAVY)
ax.legend()
plt.tight_layout()
plt.show()

# TODO: print which model has the highest AUC


### 📝 Task 5 — Optimal Threshold

The online marketing team can send vouchers to **300 customers** (15% of test set).  
Find the threshold that maximises the number of true churners caught within that budget.

In [ ]:
# ─────────────────────────────────────────────────────
# TASK 5: Optimal threshold for budget of 300 vouchers
# ─────────────────────────────────────────────────────
#
# Use your BEST model from Task 4.
#
# Approach:
#   1. Get y_prob for the best model on X_te_o
#   2. Try thresholds from 0.10 to 0.90 in steps of 0.05
#   3. For each threshold: count flagged customers, TP, FP, FN
#   4. Find the highest threshold where flagged_count <= 300
#   5. Report: how many churners are caught at that threshold?
#
# BONUS: create a plot of "churners caught" vs "vouchers sent"
#        for all thresholds (the Precision-Recall trade-off)

# TODO: implement threshold analysis

# Starter:
# for thresh in np.arange(0.10, 0.91, 0.05):
#     y_pred_t = (y_prob_best >= thresh).astype(int)
#     tn, fp, fn, tp = confusion_matrix(y_te_o, y_pred_t).ravel()
#     flagged = tp + fp
#     ...

print("Best threshold for 300-voucher budget:")
# TODO: print your result
# Expected format:
#   Threshold = X.XX → flags YYY customers, catches ZZZ / AAA churners (BB.B% recall)


### 📝 Task 6 — Learning Curves & Bias–Variance Diagnosis

Plot learning curves for your best model. Diagnose whether it has a bias or variance problem.

In [ ]:
# ─────────────────────────────────────────────────────
# TASK 6: Learning curves
# ─────────────────────────────────────────────────────
#
# Use sklearn.model_selection.learning_curve()
# Plot training AUC and validation AUC vs training set size
#
# After plotting, answer in a comment:
#   a) Does the model suffer from high bias, high variance, or neither?
#   b) Would collecting more data help? Why or why not?
#   c) What hyperparameter change would you suggest?

fig, ax = plt.subplots(figsize=(9, 5))

# TODO: use learning_curve() on your best model with X_online, y_online
# HINT:
# train_sizes, train_scores, val_scores = learning_curve(
#     best_model, X_online, y_online,
#     train_sizes=np.linspace(0.1, 1.0, 10),
#     cv=StratifiedKFold(5, shuffle=True, random_state=42),
#     scoring='roc_auc', n_jobs=-1
# )

ax.set_xlabel('Training Set Size')
ax.set_ylabel('AUC')
ax.set_title('Task 6 — Learning Curves: Online Store Best Model', fontweight='bold', color=NAVY)
ax.legend()
plt.tight_layout()
plt.show()

# TODO: write your diagnosis as Python comments here:
# Diagnosis:
# a) Bias/Variance:
# b) More data?:
# c) Suggested fix:


### 📝 Task 7 — Reflection Questions

Answer the following questions in the Markdown cells below. Write 2–4 sentences each.


**Question 1:**  
The in-store model (trained on 5,000 customers) achieved AUC = 0.87. If you applied it directly to the online dataset, what AUC would you expect, and why? What concept does this relate to?

*(Write your answer here)*


**Question 2:**  
Your manager says: "The online model has 80% accuracy — that's great, ship it!" 
Using what you know about class imbalance and AUC, explain why this might be misleading.

*(Write your answer here)*


**Question 3:**  
You train a Decision Tree with max_depth=15 on the online dataset (2,000 customers).  
Training AUC = 0.98, Validation AUC = 0.72. What is the problem, and name TWO ways to fix it.

*(Write your answer here)*


**Question 4:**  
Why is Stratified K-Fold (preserving the 25% churn ratio in each fold) more appropriate  
than regular K-Fold for this problem? What could go wrong without stratification?

*(Write your answer here)*


**Question 5 (Bonus):**  
The marketing team says: "We'd rather miss some churners than send vouchers to loyal customers."  
Should you raise or lower the classification threshold? What happens to TPR and FPR?  
Sketch (in words) what this looks like on the ROC curve.

*(Write your answer here)*


---
## 💡 Challenge — Suggested Answer Scaffolds

*Expand these cells to check your implementation after attempting the tasks.*


<details>
<summary><strong>📖 Task 1 — Dataset Generator Answer</strong></summary>

```python
def generate_online_data(n=2000, churn_rate=0.25, random_state=99):
    rng = np.random.RandomState(random_state)
    n_churn  = int(n * churn_rate)
    n_active = n - n_churn

    def make_group(size, is_churn):
        return {
            'app_sessions_month':    rng.normal(3 if is_churn else 12, 2, size).clip(0, 30),
            'delivery_rating':       rng.normal(2.5 if is_churn else 4.2, 0.6, size).clip(1, 5),
            'items_per_order':       rng.normal(8 if is_churn else 18, 4, size).clip(1, 40),
            'subscription_tier':     rng.choice([0,1,2], size=size,
                                                p=[0.7,0.25,0.05] if is_churn else [0.2,0.5,0.3]),
            'days_since_last_order': rng.normal(40 if is_churn else 10, 12, size).clip(1, 90),
            'cart_abandonment_rate': rng.beta(5, 3, size) if is_churn else rng.beta(2, 6, size),
            'support_tickets_month': rng.poisson(2 if is_churn else 0.5, size).clip(0, 10),
            'churned':               np.ones(size, int) if is_churn else np.zeros(size, int),
        }

    df = pd.concat([
        pd.DataFrame(make_group(n_churn,  True)),
        pd.DataFrame(make_group(n_active, False)),
    ]).sample(frac=1, random_state=random_state).reset_index(drop=True)
    return df
```
</details>


<details>
<summary><strong>📖 Task 5 — Threshold Answer</strong></summary>

```python
# Get probabilities from best model
y_prob_best = best_online_model.predict_proba(X_te_o)[:, 1]

results = []
for thresh in np.arange(0.10, 0.91, 0.05):
    y_pred_t = (y_prob_best >= thresh).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_te_o, y_pred_t).ravel()
    flagged = tp + fp
    recall  = tp / (tp + fn) if (tp + fn) > 0 else 0
    results.append({'threshold': thresh, 'flagged': flagged, 'tp': tp, 'recall': recall})

# Find highest threshold where flagged <= 300
budget_results = [r for r in results if r['flagged'] <= 300]
best = max(budget_results, key=lambda x: x['recall'])
print(f"Threshold = {best['threshold']:.2f} → {best['flagged']} vouchers, "
      f"{best['tp']} churners caught ({best['recall']*100:.1f}% recall)")
```
</details>


<details>
<summary><strong>📖 Reflection Answers — Key Points</strong></summary>

**Q1:** The in-store model would likely perform poorly on online customers (AUC possibly 0.55–0.70) because the feature distributions and churn drivers are completely different (in-store uses distance/spend; online uses sessions/delivery rating). This is *distribution shift* / *covariate shift*.

**Q2:** With 25% churn, a model predicting "no churn" always achieves 75% accuracy — but AUC = 0.50. Accuracy is misleading for imbalanced classes; AUC measures the quality of *ranking* churners above non-churners, which is what matters for targeted marketing.

**Q3:** This is *high variance / overfitting*. The tree memorised the 2,000 training examples. Fixes: (1) reduce max_depth to 4–6, (2) add min_samples_leaf=20, (3) switch to Random Forest which averages many trees to reduce variance.

**Q4:** Without stratification, some folds might contain only 10% churners while others have 40% — making fold-by-fold AUC scores incomparable and unstable. With only 500 churners in 2,000 records, this risk is real.

**Q5 Bonus:** Raise the threshold — model only flags customers with very high churn probability. This reduces FPR (fewer false alarms) but also reduces TPR (miss more actual churners). On the ROC curve, moving to a point further down-left — lower FPR AND lower TPR.
</details>


---
## 🎓 Concepts Summary

| Concept | What it measures | When to use |
|---|---|---|
| **Holdout** | Performance on unseen data | Quick baseline, large datasets |
| **K-Fold CV** | Stability of performance estimate | Model selection, hyperparameter tuning |
| **ROC Curve** | TPR vs FPR at every threshold | Choosing operating point for deployment |
| **AUC** | Overall discrimination ability | Comparing models; imbalanced data |
| **Bias–Variance** | Under vs. overfitting | Diagnosing why a model fails |

### Key Takeaways for RetailCo
1. **Never evaluate on training data** — use holdout or K-Fold
2. **AUC > Accuracy** for imbalanced churn problems
3. **The ROC curve is a business tool** — use it to pick your marketing threshold
4. **K-Fold reduces luck** — one holdout split can mislead you
5. **Learning curves reveal the fix** — more data vs. simpler/more-complex model

---
*📬 Questions? Submit your completed notebook to your instructor.*  
*🌟 Bonus: Try XGBoost or LightGBM and see if you can beat AUC = 0.90 on the online dataset!*
